# Autoencoders

Los **Autoencoders (AE)** se entrenan para codificar datos de entrada en un vector de características más pequeño y luego reconstruirlo mediante una segunda red neuronal llamada *decoder* (decodificador).
El vector de características se denomina **cuello de botella (*bottleneck*)** de la red, ya que buscamos comprimir los datos de entrada en una cantidad menor de características. Esta propiedad es útil en muchas aplicaciones, especialmente en la compresión de datos o en la comparación de imágenes usando métricas más allá de las comparaciones a nivel de píxel.

## Construiendo un Autoencoder

Un autoencoder aprende a reproducir su entrada pasando por un **cuello de botella** que obliga a una representación compacta. Un *encoder* $f_\theta$ mapea una entrada $\mathbf{x}$ a un código latente $\mathbf{z}$, y un *decoder* $g_\phi$ mapea $\mathbf{z}$ de nuevo a una reconstrucción $\hat{\mathbf{x}}$:

$$
\mathbf{z}=f_\theta(\mathbf{x}),\qquad \hat{\mathbf{x}}=g_\phi(\mathbf{z}).
$$

El entrenamiento minimiza un objetivo de reconstrucción

$$
\mathcal{L}(\theta,\phi)=\mathbb{E}_{\mathbf{x}}\big[\ell(\hat{\mathbf{x}},\mathbf{x})\big].
$$

La elección de $\ell$ debe corresponderse con el tipo de datos y el modelo de ruido asumido:

* **Error cuadrático medio (MSE):** corresponde a verosimilitud Gaussiana para datos continuos.
* **Entropía cruzada binaria:** adecuada para observaciones Bernoulli.
* **Entropía cruzada categórica:** para datos discretos.
* **Pérdida de Poisson:** común para datos de conteo.

El objetivo es que $\mathbf{z}$ preserve la información necesaria para reconstruir $\mathbf{x}$ eliminando redundancias.

Arquitectónicamente, los autoencoders se adaptan a la modalidad:

* Para vectores tabulares: $f_\theta$ y $g_\phi$ suelen ser perceptrones multicapa.
* Para imágenes: se emplean redes convolucionales profundas que reducen progresivamente la resolución espacial mientras aumentan la capacidad de canales.

Un diseño práctico aplica una secuencia de convoluciones con *stride* para reducir la entrada tres veces, disminuyendo alto y ancho por un factor 2 en cada paso. El mapa resultante se aplana y proyecta a un vector $\mathbf{z}\in\mathbb{R}^d$. La dimensión $d$ define la fuerza del cuello de botella:

* $d$ pequeño: compresión fuerte y aprendizaje de estructuras compactas.
* $d$ grande: reconstrucción más sencilla pero riesgo de transmitir detalles innecesarios.

El *decoder* generalmente refleja este proceso en reversa, expandiendo $\mathbf{z}$ hasta la forma original usando convoluciones transpuestas (*deconvolutions*).

![autoencoder](./images/autoencoders/autoencoders.png)

En AEs rara vez se aplica **Batch Normalization**, porque se desea que la codificación de cada imagen sea independiente de las demás. Si se normalizara por lotes, se introducirían correlaciones no deseadas en la codificación o decodificación. En algunos casos se usa como regularización, pero lo más recomendable es recurrir a técnicas como *Instance Normalization* o *Layer Normalization*.



### Imágenes Fuera de Distribución

Podemos explorar limitaciones: ¿qué ocurre si intentamos reconstruir una imagen fuera de la distribución del dataset? El *decoder* habrá aprendido patrones comunes y puede fallar en reconstruir imágenes que no los sigan.

En general, los autoencoders fallan en reconstruir **ruido de alta frecuencia** (cambios bruscos en pocos píxeles) debido al uso de MSE. Pequeños desajustes en el *decoder* producen grandes pérdidas, por lo que el modelo tiende a promediar en esas regiones. En cambio, para ruido de baja frecuencia, un pequeño desajuste no altera demasiado la imagen. Cuanto mayor sea la dimensionalidad latente, más de este ruido de alta frecuencia podrá reconstruirse.



## Autoencoders Variacionales (VAE)

Los **VAEs** son una versión generativa y **probabilística** de los autoencoders. Un VAE combina:

* Un *decoder* $p_\theta(\mathbf{x}\mid \mathbf{z})$ que genera datos desde un vector latente $\mathbf{z}$.
* Un *encoder* $q_\phi(\mathbf{z}\mid \mathbf{x})$ que aproxima la distribución posterior sobre los latentes.

Se asume un prior simple $p(\mathbf{z})=\mathcal{N}(\mathbf{0},\mathbf{I})$, que regula el espacio latente hacia una distribución Gaussiana, a diferencia de los AEs clásicos donde no hay restricciones. Para generar, se toma $\mathbf{z}\sim p(\mathbf{z})$ y se decodifica $\mathbf{x}\sim p_\theta(\mathbf{x}\mid \mathbf{z})$.


### La ELBO

La identidad variacional detrás de los VAEs es

$$
\log p_\theta(\mathbf{x})
=\underbrace{\mathbb{E}_{q_\phi(\mathbf{z}\mid\mathbf{x})}\!\big[\log p_\theta(\mathbf{x}\mid\mathbf{z})\big]
-\mathrm{KL}\!\big(q_\phi(\mathbf{z}\mid\mathbf{x})\,\|\,p(\mathbf{z})\big)}_{\displaystyle \mathcal{L}(\theta,\phi;\mathbf{x})\;\text{(ELBO)}}
\;+\;\underbrace{\mathrm{KL}\!\big(q_\phi(\mathbf{z}\mid\mathbf{x})\,\|\,p_\theta(\mathbf{z}\mid\mathbf{x})\big)}_{\ge 0}.
$$

Dado que el último término es no negativo, $\mathcal{L}$ es una **cota inferior** de la log-verosimilitud exacta. Maximizar la ELBO incrementa la verosimilitud de los datos y acerca el posterior aproximado $q_\phi(\mathbf{z}\mid\mathbf{x})$ al posterior verdadero $p_\theta(\mathbf{z}\mid\mathbf{x})$; la cota se vuelve ajustada cuando ambos coinciden.

Maximizar la log-verosimilitud exacta $\log p_\theta(\mathbf{x})$ es intratable, por lo que los VAEs maximizan la **evidence lower bound (ELBO)**:

$$
\mathcal{L}(\theta,\phi;\mathbf{x})
\;=\;
\underbrace{\mathbb{E}_{q_\phi(\mathbf{z}\mid \mathbf{x})}[\log p_\theta(\mathbf{x}\mid \mathbf{z})]}_{\text{reconstrucción}}
\;-\;
\underbrace{\mathrm{KL}\!\left(q_\phi(\mathbf{z}\mid \mathbf{x})\,\|\,p(\mathbf{z})\right)}_{\text{regularización}}.
$$

* El primer término fuerza a que las reconstrucciones se parezcan a los datos.
* El segundo mantiene los latentes inferidos cerca del prior, dando lugar a un espacio latente bien estructurado.

La conexión con las pérdidas de reconstrucción comunes proviene de la elección de la verosimilitud $p_\theta(\mathbf{x}\mid\mathbf{z})$:

* Si $p_\theta(\mathbf{x}\mid\mathbf{z})$ es Bernoulli factorada (típico para datos en $[0,1]$):

$$
\log p_\theta(\mathbf{x}\mid\mathbf{z})
=\sum_i x_i\log \hat{x}_i+(1-x_i)\log(1-\hat{x}_i),
$$

con $\hat{x}=\sigma(f_\theta(\mathbf{z}))$.
El negativo de esto es exactamente la entropía cruzada binaria (BCE). Así, maximizar el término de reconstrucción equivale a minimizar la BCE, salvo por la expectativa sobre $q_\phi(\mathbf{z}\mid\mathbf{x})$ (normalmente aproximada por una o pocas muestras).

* Si $p_\theta(\mathbf{x}\mid\mathbf{z})$ es Gaussiana con varianza fija $\sigma^2 I$ y media $\mu_\theta(\mathbf{z})$:

$$
\log p_\theta(\mathbf{x}\mid\mathbf{z})
=-\frac{1}{2\sigma^2}\|\mathbf{x}-\mu_\theta(\mathbf{z})\|_2^2 - \tfrac{D}{2}\log(2\pi\sigma^2),
$$

entonces maximizar el término de reconstrucción equivale a minimizar el error cuadrático medio (MSE), de nuevo salvo por una constante y un factor $1/(2\sigma^2)$.
Cuando $\sigma^2$ se aprende (Gaussiana heterocedástica), la reconstrucción se convierte en un *MSE ponderado* más una penalización sobre la log-varianza predicha.

> En resumen, el objetivo de un VAE es *la pérdida de reconstrucción negativa (BCE o MSE, según la verosimilitud asumida) más una penalización KL en los latentes*. La elección entre BCE o MSE no es arbitraria: codifica la suposición sobre el modelo de ruido de las observaciones — tipo Bernoulli para datos binarios/normalizados, Gaussiano para datos continuos con ruido aditivo.

### Reparametrización

Para retropropagar a través del muestreo:

$$
\mathbf{z}=\boldsymbol{\mu}_\phi(\mathbf{x})+\boldsymbol{\sigma}_\phi(\mathbf{x})\odot\boldsymbol{\epsilon},
\quad \boldsymbol{\epsilon}\sim\mathcal{N}(\mathbf{0},\mathbf{I}).
$$

Así se convierte la muestra estocástica en una función determinista de $\mathbf{x}$ y ruido.



### ¿Qué producen las redes?

* **Encoder $q_\phi(\mathbf{z}\mid \mathbf{x})$:** da $\boldsymbol{\mu}*\phi(\mathbf{x})$ y $\boldsymbol{\sigma}*\phi(\mathbf{x})$.
* **Decoder $p_\theta(\mathbf{x}\mid \mathbf{z})$:** produce parámetros de una distribución adecuada (Gaussiana, Bernoulli, etc.).



### El Entrenamiento en la práctica

Para cada *minibatch*:

1. Codificar $\mathbf{x}\rightarrow (\mu, \sigma)$.
2. Muestrear $\mathbf{z}=\mu+\sigma\odot \epsilon$.
3. Decodificar y calcular ELBO.
4. Actualizar parámetros con gradiente de $-\mathcal{L}$.

Se reporta ELBO o -ELBO. En imágenes, es común la métrica **bits per dimension (bpd)**: 
$$
\text{bpd} = -\log_2 p_\theta(\mathbf{x}) / (H\!\times\!W\!\times\!C).
$$



### Problemas comunes y soluciones

* **Colapso del posterior:** usar *KL warm-up*, limitar la capacidad del *decoder*, o aplicar *free bits*.
* **Datos discretos:** aplicar *dequantization* (ruido uniforme) o distribuciones discretizadas.
* **Calibración:** evaluar con ELBO en *held-out set* y calidad de muestras.



### VAEs y PCA Probabilístico (PPCA)

El **PPCA** reformula el PCA clásico como un modelo generativo con incertidumbre:

$$
x \approx Wz+\mu+\varepsilon, \quad \varepsilon\sim \mathcal{N}(0,\sigma^2 I).
$$

Aquí $z\sim \mathcal{N}(0,I)$ es un latente, y $x$ se genera linealmente con ruido Gaussiano.

* El prior de $z$ en VAE ($\mathcal{N}(0,I)$) coincide con PPCA.
* El *decoder* del VAE se reduce al lineal de PPCA con varianza isotrópica.
* El *encoder* del VAE corresponde al posterior Gaussiano exacto de PPCA.

En este caso, la ELBO = log-verosimilitud exacta, y el VAE recupera la solución de PPCA.

Reemplazando el *decoder* lineal por redes neuronales, el VAE se convierte en una versión no lineal de PPCA, sacrificando interpretabilidad y exactitud cerrada a cambio de mayor flexibilidad.
